# Practical 7: Text Classification Model

**Concept:** **Text classification** assigns a predefined category (label) to a free-text document — e.g. spam/ham, news topic, sentiment. It is one of the most common applications of NLP.

The standard recipe (already used in Practical 5) scales here to a real dataset:
1. Load a labeled text dataset
2. Convert documents to **TF-IDF** features
3. Train a classifier — **Naive Bayes** and **Linear SVM**
4. Evaluate with **accuracy** and a **classification report**

We use the classic **20 Newsgroups** dataset, which contains ~20,000 messages grouped into 20 news topics. To keep it fast, we classify just three related-looking topics.

## Import Required Libraries

In [1]:
import numpy as np

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Load the Dataset
`fetch_20newsgroups` downloads a small subset of the dataset (3 categories) and already splits it into `train` and `test` sets. The `subset` parameter controls this.

In [2]:
# Choose 3 topics that overlap in vocabulary, making the task non-trivial
categories = ["sci.space", "comp.graphics", "rec.sport.baseball"]

# Load the train and test sets
newsgroups_train = fetch_20newsgroups(
    subset="train", categories=categories, remove=("headers", "footers", "quotes")
)
newsgroups_test = fetch_20newsgroups(
    subset="test", categories=categories, remove=("headers", "footers", "quotes")
)

print("Dataset Loaded")
print("-" * 50)
print(f"Categories: {newsgroups_train.target_names}")
print(f"Training samples: {len(newsgroups_train.data)}")
print(f"Testing samples:  {len(newsgroups_test.data)}")

Dataset Loaded
--------------------------------------------------
Categories: ['comp.graphics', 'rec.sport.baseball', 'sci.space']
Training samples: 1774
Testing samples:  1180


## Explore the Data
Let's look at one example document and its label.

In [3]:
print("Sample Document")
print("-" * 50)
print(newsgroups_train.data[0])
print("\nLabel:", newsgroups_train.target_names[newsgroups_train.target[0]])

Sample Document
--------------------------------------------------

By '8 grey level images' you mean 8 items of 1bit images?
It does work(!), but it doesn't work if you have more than 1bit
in your screen and if the screen intensity is non-linear.

With 2 bit per pixel; there could be 1*c_1 + 4*c_2 timing,
this gives 16 levels, but they are linear if screen intensity is
linear.
With 1*c_1 + 2*c_2 it works, but we have to find the best
compinations -- there's 10 levels, but 16 choises; best 10 must be
chosen. Different compinations for the same level, varies a bit, but
the levels keeps their order.

Readers should verify what I wrote... :-)

Label: comp.graphics


## Convert Text to TF-IDF Features
The `TfidfVectorizer` handles lowercasing, tokenization, stopword removal (`stop_words="english"`) and TF-IDF weighting in one step. We **fit** it on the training data and only **transform** the test data, so the test set is never leaked into the vocabulary.

In [4]:
# Build the vectorizer and fit on training data only
vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
X_train = vectorizer.fit_transform(newsgroups_train.data)
X_test = vectorizer.transform(newsgroups_test.data)

# Labels
y_train = newsgroups_train.target
y_test = newsgroups_test.target

print("TF-IDF Feature Matrices")
print("-" * 50)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

TF-IDF Feature Matrices
--------------------------------------------------
X_train shape: (1774, 10000)
X_test shape:  (1180, 10000)


## Train a Multinomial Naive Bayes Classifier
**Naive Bayes** is the classic text classification baseline. Despite the "naive" independence assumption, it works surprisingly well on text.

In [5]:
# Train Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Predict on the test set
nb_pred = nb_model.predict(X_test)

nb_accuracy = accuracy_score(y_test, nb_pred)
print(f"Naive Bayes Test Accuracy: {nb_accuracy:.4f} ({nb_accuracy*100:.2f}%)")

Naive Bayes Test Accuracy: 0.9144 (91.44%)


## Train a Linear SVM Classifier
**Support Vector Machines (SVM)** often outperform Naive Bayes on text. `LinearSVC` is a fast linear SVM that works well with high-dimensional TF-IDF features.

In [6]:
# Train Linear SVM
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train, y_train)

# Predict on the test set
svm_pred = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_pred)
print(f"Linear SVM Test Accuracy: {svm_accuracy:.4f} ({svm_accuracy*100:.2f}%)")

Linear SVM Test Accuracy: 0.9000 (90.00%)


## Classification Report (SVM)
The **classification report** shows per-class precision, recall and F1-score. This tells us *which* categories the model confuses.

In [7]:
print("SVM Classification Report")
print("-" * 50)
print(classification_report(
    y_test, svm_pred,
    target_names=newsgroups_test.target_names
))

SVM Classification Report
--------------------------------------------------
                    precision    recall  f1-score   support

     comp.graphics       0.93      0.90      0.91       389
rec.sport.baseball       0.88      0.94      0.91       397
         sci.space       0.90      0.86      0.88       394

          accuracy                           0.90      1180
         macro avg       0.90      0.90      0.90      1180
      weighted avg       0.90      0.90      0.90      1180



## Confusion Matrix
The **confusion matrix** visually shows which categories get mixed up. Off-diagonal entries are misclassifications.

In [8]:
# Compute the confusion matrix
cm = confusion_matrix(y_test, svm_pred)

print("Confusion Matrix (rows = true, columns = predicted)")
print("-" * 50)

names = newsgroups_test.target_names
print(f"{'':14}" + "".join(f"{n[:10]:>11}" for n in names))

for i, name in enumerate(names):
    row = "".join(f"{cm[i][j]:>11}" for j in range(len(names)))
    print(f"{name[:13]:14}" + row)

Confusion Matrix (rows = true, columns = predicted)
--------------------------------------------------
               comp.graph rec.sport.  sci.space
comp.graphics         349         18         22
rec.sport.bas           5        375         17
sci.space              22         34        338


## Compare the Models
We compare the two classifiers side by side.

In [9]:
print("Model Comparison")
print("-" * 50)
print(f"Multinomial Naive Bayes  -> {nb_accuracy:.4f}")
print(f"Linear SVM               -> {svm_accuracy:.4f}")

Model Comparison
--------------------------------------------------
Multinomial Naive Bayes  -> 0.9144
Linear SVM               -> 0.9000


## Classifying New Documents
Finally, we feed brand-new documents to the SVM model and let it predict the newsgroup.

In [10]:
new_docs = [
    "The shuttle will be launched into orbit around the moon tomorrow.",
    "How do I render 3D graphics with OpenGL on a laptop?",
    "The pitcher threw a fastball and the batter hit a home run."
]

# Vectorize the new documents with the same fitted vectorizer
new_vectors = vectorizer.transform(new_docs)
new_predictions = svm_model.predict(new_vectors)

print("Predictions on New Documents")
print("-" * 50)

for doc, pred in zip(new_docs, new_predictions):
    category = newsgroups_test.target_names[pred]
    print(f"Document: {doc}")
    print(f"Predicted category: {category}")
    print()

Predictions on New Documents
--------------------------------------------------
Document: The shuttle will be launched into orbit around the moon tomorrow.
Predicted category: sci.space

Document: How do I render 3D graphics with OpenGL on a laptop?
Predicted category: comp.graphics

Document: The pitcher threw a fastball and the batter hit a home run.
Predicted category: rec.sport.baseball



## Summary

In this practical we built a complete text classification pipeline:
1. Loaded the **20 Newsgroups** dataset (3 categories) with a built-in train/test split.
2. Vectorized the documents with **TF-IDF** (fit on train only).
3. Trained **Multinomial Naive Bayes** and **Linear SVM** classifiers.
4. Evaluated with **accuracy**, a **classification report** and a **confusion matrix**.
5. Classified new unseen documents.

The same TF-IDF + classifier pipeline is the foundation of many real-world NLP products: spam filters, review classification, topic detection and more.